# Recovery

Wilson & Collins's (2019) parameter and model recovery, with the tests' power and false
rejections along R and n_rep ([`README.md`](README.md) § Recovery; [`schedule.md`](schedule.md)
§ Recovery), before the main runs, at H (`config.NETWORK_MAX_ROUNDS`). This notebook sets the
ranges (§ 3) and reads the results (§§ 5–7); [`recovery.py`](recovery.py) computes them, the
datasets as an SCC array (§ 4). A placeholder: each section runs once its inputs exist and prints
what it waits for otherwise. Results go to `logs/recovery/` ([`schedule.md`](schedule.md)
§ Recovery, Files).

## 1. Inputs

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import numpy as np
import pandas as pd

pd.set_option("display.precision", 3)  # every table at three decimals

CWD = Path.cwd().resolve()
REPO = next(p for p in [CWD, *CWD.parents] if (p / "simulation" / "config.py").is_file())
sys.path.insert(0, str(REPO / "analysis"))
import fit  # noqa: E402
import config  # noqa: E402  (fit puts simulation/ on the path)

import recovery  # noqa: E402
from metrics import wilson  # noqa: E402
from surrogate import LIMIT, Pool  # noqa: E402

AUDIT = REPO / "analysis" / "logs" / "audit"
TOPICS = ("nuclear", "renewables", "coffee_tea")
MODELS = ("qwen3.5-nodraft:4b", "gemma4-nodraft:e4b", "granite4.2-nodraft:3b")
CELLS = [f"{t}_{m.replace(':', '-')}" for t in TOPICS for m in MODELS]
RECOVERY_CELLS = [c for c in CELLS if c not in recovery.SKIPPED]   # § 3 on
H = config.NETWORK_MAX_ROUNDS    # fixed by design (schedule.md § Runs, Horizon)
SEEDS = range(1, 301)        # the benchmarks' seeds
R = 33                       # recovery's R, read also at 17 and 9 (recovery.STEPS)
N_REP = 40                   # replicates per run and model, read also at 20, 10, 5 (recovery.HALVINGS)
N_DATA = 100                 # datasets per cell and generating model
JOBS = 8                     # 16 workers ran out of memory on import, on 7 GB of RAM; 8 did not

# A cell counts once its audit phase is in; calibration alone holds no pool.
POOLS = {c: Pool.from_audit(AUDIT / c) for c in CELLS
         if (AUDIT / c / "audit_meetings").is_dir()}
ALL_POOLS = len(POOLS) == len(CELLS)
SETTINGS = recovery.LOGS / "settings.json"     # the array's input, copied to the SCC
PROFILES = recovery.LOGS / "profiles.csv"
DATASETS = recovery.LOGS / "datasets.jsonl"    # merged by the array (RUNBOOK § Recovery on the CPUs)
PARAMS = {"H": H, "seeds": SEEDS}   # the Limit curves' sidecar, beside its commit
print("audit cells:", len(POOLS), "of", len(CELLS), "| settings:", SETTINGS.is_file(),
      "| datasets:", len(recovery.read_rows(DATASETS)))

## 2. The Limit on each cell's replayed exchanges

Not a check: the ladder's curve ([`schedule.md`](schedule.md) § Sensitivity;
[`README.md`](README.md) § The references). D̄(p_inter, t) from the surrogate under the
Limit's rule on each cell's replayed exchanges, from the polarized start, over seeds 1–300, to
T = 20,000.

In [ ]:
if ALL_POOLS:
    for c, p in POOLS.items():
        recovery.save_curve(recovery.LOGS / c / "limit_divergence.csv",
                            recovery.divergence_curve(LIMIT, p, SEEDS, jobs=JOBS), PARAMS | {"cell": c})
else:
    print("waits for the audit in", sorted(set(CELLS) - set(POOLS)))

## 3. Step 1: the ranges

Per cell, b, the name intercepts and κ from M0 on the audit's empty-memory declarations (`recovery.baseline`); then each
weight alone, the others at 0, by powers of 2 (`recovery.cell_ranges`): from its onset, the largest
value that moves none of the three behaviour summaries by more than 5% of its scale from the weight
at 0, to its saturation, the first value past it whose doubling moves none. One dataset of R runs
per value, the same seed throughout. Weights and rates are drawn log-uniformly, rates from
`recovery.RATE_LOW`, the rate whose half-life is the 2H entries an agent holds at H, to 1. A cell
whose saturation aborts is named in `recovery.SKIPPED`. The settings file is the array's one input;
delete it to redo this section.

In [ ]:
if SETTINGS.is_file():
    settings = json.loads(SETTINGS.read_text(encoding="utf-8"))
    profiles = pd.read_csv(PROFILES)
elif ALL_POOLS:
    cells, rows = {}, []
    for c in RECOVERY_CELLS:
        fixed = recovery.baseline(AUDIT / c)
        ranges, prof = recovery.cell_ranges(fixed, POOLS[c], R, H)
        cells[c] = {"ranges": ranges}
        rows += [{"cell": c, **x} for x in prof]
    recovery.save(PROFILES, rows, {"H": H, "R": R})
    settings = recovery.write_settings(SETTINGS, cells, H, R, N_REP, N_DATA)
    profiles = pd.DataFrame(rows)
else:
    settings = None
    print("waits for the audit in", sorted(set(CELLS) - set(POOLS)))
if settings:
    print("tasks:", recovery.n_tasks(settings))
    display(pd.DataFrame({c: {k: "{:g} – {:g}".format(*s["ranges"]["M3"][k]) for k in recovery.WEIGHTS}
                          for c, s in settings["cells"].items()}).T)   # each weight's onset – saturation
    fig, axes = plt.subplots(1, len(recovery.WEIGHTS), figsize=(6, 3), sharey=True)
    for ax, ((m, w), of) in zip(axes, profiles.groupby(["model", "parameter"], sort=False)):
        low = of.value[of.value > 0].min() / 2   # where 0 is drawn on the log axis
        for c, x in of.groupby("cell"):
            ax.plot(x.value.where(x.value > 0, low), x.divergence, marker=".", label=c)
        ax.set(xscale="log", title=f"{m} {w}", xlabel="weight (0 at the left end)")
        ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,g}"))   # decimal labels on log axes, not 10^n
    axes[0].set_ylabel("mean D")
    plt.show()

## 4. Steps 2–4: the datasets, on the SCC

`N_DATA` datasets per cell and generating model, one array task each
([`run_recovery.sh`](run_recovery.sh); RUNBOOK § Recovery on the CPUs): 33 surrogate runs on the
even p_inter grid from the polarized start; at R = 33, 17 and 9 the four models' pooled fits, one
leave-one-run-out pass of each, the tests H1 and H3 and, from 40 replicates per run of M1 and
M3 read at 40, 20, 10 and 5, H2. `recovery.py merge` writes `DATASETS`, copied back.

In [ ]:
rows = recovery.read_rows(DATASETS)
if rows:
    df = pd.DataFrame(rows)
    print(len(rows), "datasets;", df.groupby(["cell", "true"]).size().describe()[["min", "max"]].to_dict(),
          "per cell and model; nodes:", df.node.nunique())
else:
    print("waits for the array's merged rows at", DATASETS)

## 5. Step 3: parameter recovery

Per cell, simulated against recovered per model and parameter at each R, with r and mean bias;
rates and weights on the log scale they are drawn on, bias a mean log ratio, a fit at or below 0
counted in `nonpositive` and off the axes; b, κ and the name intercepts, fixed per cell, bias alone. Then what the fit
adds to the correlations among the drawn parameters, which the draws leave near 0.

In [ ]:
if rows:
    long = pd.DataFrame(recovery.parameter_rows(rows, R))
    long = long[long.groupby(["cell", "true_model", "parameter"]).simulated.transform("nunique") > 1]
    panels = list(long[["true_model", "parameter"]].drop_duplicates().itertuples(index=False))
    cells = sorted(long.cell.unique())
    fig, axes = plt.subplots(len(cells), len(panels), squeeze=False,
                             figsize=(2 * len(panels), 2 * len(cells)))
    for i, c in enumerate(cells):
        for j, (m, k) in enumerate(panels):
            of = long[(long.cell == c) & (long.true_model == m) & (long.parameter == k)]
            axes[i, j].scatter(of.simulated, of.recovered, s=4)
            if k in recovery.LOG_DRAWN:   # a fit at or below 0 falls off, counted in the table
                axes[i, j].set(xscale="log", yscale="log")
                for axis in (axes[i, j].xaxis, axes[i, j].yaxis):
                    axis.set_major_formatter(StrMethodFormatter("{x:,g}"))   # decimal labels, not 10^n
            axes[i, j].set_title(f"{c}\n{m} {k}", fontsize=7)
    plt.tight_layout()
    plt.show()
    display(pd.concat({n: pd.DataFrame(recovery.parameter_recovery(rows, n))
                       .set_index(["cell", "true_model", "parameter"])
                       for n in recovery.sizes(rows[0])}, axis=1))
    display(pd.DataFrame(recovery.induced_correlations(rows, R)).set_index(["cell", "true_model"]))
    print("fits at a bound:", pd.Series([f"{r['cell']} {k}" for r in rows for k in r[f"R{R}"]["at_bound"]],
                                        dtype=object).value_counts().to_dict())
else:
    print("waits for § 4")

## 6. Step 4: model recovery

Per cell, P(chosen | true) and its inversion P(true | chosen) at each R, on the held-out score the
tests use and on BIC, the same pooled fits. Reading them against the hypotheses:
[`README.md`](README.md) § Recovery.

In [ ]:
if rows:
    tables = {}
    for c in sorted({r["cell"] for r in rows}):
        of = [r for r in rows if r["cell"] == c]
        for n in recovery.sizes(rows[0])[::-1]:
            for key in ("chosen", "chosen_bic"):
                table, inverse = recovery.confusion([(r["true"], r[f"R{n}"][key]) for r in of])
                tables[(c, n, key, "P(chosen | true)")] = pd.DataFrame(table).T
                tables[(c, n, key, "P(true | chosen)")] = pd.DataFrame(inverse).T
    display(pd.concat(tables, names=["cell", "R", "criterion", "table", "given"]).fillna(0))
else:
    print("waits for § 4")

## 7. Power and false rejections

Per cell, each test's rejection share at its level (`recovery.rejection_rows`), where its hypothesis
holds (power) and where it does not (false rejection; `recovery.truth`), with Wilson intervals; the
n_rep curve, flat from the n_rep the runs need; the normal approximation `recovery.normal_power`
predicting R = 33 from 17 and 9, then beyond; the declaration gains against p_inter.

In [ ]:
if rows:
    rej = pd.DataFrame(recovery.rejection_rows(rows))
    at = rej[rej.n_rep == N_REP]

    def share(x):
        lo, hi = wilson(int(x.sum()), len(x))
        return pd.Series({"share": x.mean(), "lo": lo, "hi": hi, "n": len(x)})

    print("false rejections (holds False), at n_rep", N_REP)
    display(at[at.holds == False].groupby(["cell", "test", "R"]).rejects.apply(share).unstack())  # noqa: E712
    print("power (holds True), at n_rep", N_REP)
    display(at[at.holds == True].groupby(["cell", "test", "R"]).rejects.apply(share).unstack())  # noqa: E712
    b = rej[(rej.holds == True) & rej.test.eq("H2")]  # noqa: E712
    print("the n_rep curve: power of H2 by n_rep")
    display(b.pivot_table("rejects", ["cell", "test", "R"], "n_rep"))

    tests = ("H1", "H2")
    pred = []
    for c in sorted({r["cell"] for r in rows}):
        for h in tests:
            of = [r for r in rows if r["cell"] == c and recovery.truth(r["true"], h)]
            for n in recovery.sizes(rows[0]):
                g = [r[f"R{n}"]["gains"][h] for r in of]
                pred += [{"cell": c, "test": h, "from R": n, "at R": t, "power": recovery.normal_power(g, t)}
                         for t in (R, 65, 129)]   # R - 1 doubled, twice
    print("normal approximation (observed at R = 33 above)")
    display(pd.DataFrame(pred).pivot_table("power", ["cell", "test", "from R"], "at R"))

    grid = recovery.grid(R)
    gains = pd.DataFrame([{"cell": r["cell"], "true": r["true"], "test": h, "p_inter": p, "gain": g}
                          for r in rows for h in ("H1", "M1:M0", "M2:M0")
                          for p, g in zip(grid, r[f"R{R}"]["gains"][h])])
    cells = sorted(gains.cell.unique())
    fig, axes = plt.subplots(len(cells), 3, figsize=(15, 3 * len(cells)), sharex=True, squeeze=False)
    for i, c in enumerate(cells):
        for j, (h, of) in enumerate(gains[gains.cell == c].groupby("test")):
            of.groupby(["true", "p_inter"]).gain.mean().unstack(0).plot(
                ax=axes[i, j], title=f"{c} {h}", legend=i == 0 and j == 0)
    plt.tight_layout()
    plt.show()
else:
    print("waits for § 4")

## 8. Steps 5–6: what follows

- A failure is a bug until shown otherwise, then a design that gives too little data: raise R or change
  the models, and repeat.
- Where a rate does not come back in a cell, only the products γ_Kα_K and βα are read there.
- R comes from the run budget, with each cell's power at it reported from § 7; n_rep
  is the smallest from which the n_rep curve is flat.
- After the runs, a cell whose fits leave its ranges repeats § 4 for that cell over the fits' range,
  and a cell whose lean on the runs departs from the audit's with the runs' baseline
  ([`README.md`](README.md) § The fit, Range).